# Phishing Email Detection: Handcrafted Rules vs. Learned Text Features

This notebook compares two ways of detecting phishing emails: six rule-style features (urgency words, URLs, etc.) versus TF-IDF text features learned from the data. It also measures how much phishing slips past keyword-based detection entirely.

## Stage 0: Setup and Inspection

Load the raw dataset and confirm its basic shape, class balance, missing values, duplicates, and length outliers before changing anything. Knowing these numbers up front lets us report exactly what cleaning removed later.

In [ ]:
import pandas as pd

# Paths are relative to the notebooks/ folder
DATA_PATH = "../data/phishing_emails.csv"

# The first CSV column is an unnamed row index, so use it as the index instead of a data column
raw = pd.read_csv(DATA_PATH, index_col=0)

print("Shape (rows, columns):", raw.shape)
print("Columns:", list(raw.columns))

In [ ]:
# Class balance: how many emails of each type
print(raw["Email Type"].value_counts())

# Missing email bodies
print("\nNull Email Text values:", raw["Email Text"].isna().sum())

# Duplicate email bodies (counts every repeat after the first occurrence; nulls excluded)
non_null_text = raw["Email Text"].dropna()
print("Duplicate Email Text values:", non_null_text.duplicated().sum())

In [ ]:
# Length of each email in characters, to spot extreme outliers
lengths = non_null_text.str.len()
print(lengths.describe())

# The five longest emails, with their label
longest = lengths.sort_values(ascending=False).head(5)
print("\nLongest emails:")
print(pd.DataFrame({"length": longest, "Email Type": raw.loc[longest.index, "Email Type"]}))

In [ ]:
# Show the first few rows, cutting each email body to 200 characters so outputs stay readable
preview = raw.head(5).copy()
preview["Email Text"] = preview["Email Text"].str.slice(0, 200) + "..."
pd.set_option("display.max_colwidth", 210)
preview